# MONITORAMENTO PIPELINE CAMADA SILVER

**Pipeline:** Análise da Alfabetização no Brasil


**Objetivo:**

 - Monitorar falhas de leitura

 - Medir latência

 - Monitorar volume processado

 - Validar qualidade dos dados

 - Gerar alertas

 - Verificação de duplicidade

 - Detecção de valores ausentes
 
 - Validação de chaves de relacionamento

## 1. CONFIGURAÇÃO DE LOG

Configura o sistema de logs do pipeline, registrando eventos, métricas e erros em um arquivo para fins de monitoramento

In [1]:
import logging

logging.basicConfig(
filename="monitoramento_silver.log",
level=logging.INFO,
format="%(asctime)s - %(levelname)s - %(message)s"
)

StatementMeta(sparkpool, 33, 2, Finished, Available, Finished, False)

## 2. Configuração de Acesso

Conexão ao ADLS Gen2 via **Azure Key Vault** — a chave de acesso é lida em runtime e nunca fica exposta no código (boa prática de segurança).

In [2]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# A chave é lida do Key Vault em runtime — nunca fica exposta no código
# ============================================================
from notebookutils import mssparkutils
from pyspark.sql.functions import col
import time
from datetime import datetime

CONTA_STORAGE = "stalfalfabetizacao"

# Busca a chave de acesso direto do Key Vault
CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

# Configura o Spark para usar a chave
spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

# Caminhos base de cada camada
CAMINHO_BRONZE = f"abfss://bronze@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_GOLD   = f"abfss://gold@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")
print(f"  Bronze : {CAMINHO_BRONZE}")
print(f"  Silver : {CAMINHO_SILVER}")
print(f"  Gold   : {CAMINHO_GOLD}")

StatementMeta(sparkpool, 33, 3, Finished, Available, Finished, False)

Conexão configurada com sucesso!
  Bronze : abfss://bronze@stalfalfabetizacao.dfs.core.windows.net/
  Silver : abfss://silver@stalfalfabetizacao.dfs.core.windows.net/
  Gold   : abfss://gold@stalfalfabetizacao.dfs.core.windows.net/


In [3]:
# ====================================================
# LEITURA DOS DATASETS
# ====================================================


datasets = {
    "uf": "uf/",
    "municipio": "municipio/",
    "meta_brasil": "meta_brasil/",
    "meta_por_uf": "meta_por_uf/",
    "meta_por_municipio": "meta_por_municipio/",
    "ts_municipio": "ts_municipio/",
    "ts_estado": "ts_estado/"
}

for nome, caminho in datasets.items():

    inicio_dataset = time.time()

    logging.info(
        f"Lendo dataset: {nome}"
    )

    df = spark.read.parquet(
        CAMINHO_SILVER + caminho
    )

      # Monitoramento de Volume
    volume = df.count()

    logging.info(
        f"{nome} -> {volume} registros"
    )

    print(
        f"{nome}: {volume} registros"
    )

StatementMeta(sparkpool, 33, 4, Finished, Cancelled, Cancelled, False)

## 3. Monitoramento

O código monitora a qualidade e a performance dos datasets da camada Silver, verificando falhas de ingestão, valores nulos, duplicidades, relacionamentos entre tabelas e latência de processamento, além de registrar o tempo total de execução do pipeline.

In [ ]:
from pyspark.sql.functions import (
    col,
    when,
    sum as spark_sum
)

# ====================================================
# CONFIGURAÇÕES
# ====================================================

CHAVES = {

    "uf": [
        "ano",
        "sigla_uf",
        "serie",
        "rede"
    ],

    "municipio": [
        "ano",
        "id_municipio",
        "serie",
        "rede"
    ],

    "meta_brasil": [
        "ano",
        "rede"
    ],

    "meta_por_uf": [
        "ano",
        "sigla_uf",
        "rede"
    ],

    "meta_por_municipio": [
        "ano",
        "id_municipio",
        "rede"
    ],

    "ts_estado": [
        "ano",
        "sigla_uf",
        "TP_SERIE",
        "ID_TIPO_REDE"
    ],

    "ts_municipio": [
        "ano",
        "id_municipio",
        "TP_SERIE",
        "ID_TIPO_REDE"
    ]
}

# ====================================================
# INÍCIO PIPELINE
# ====================================================

inicio_pipeline = time.time()

# ====================================================
# MONITORAMENTO DOS DATASETS
# ====================================================

for nome, caminho in datasets.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {nome}")
    print("=" * 70)

    inicio_dataset = time.time()

    # =================================================
    # FALHA DE INGESTÃO
    # =================================================

    try:

        df = (
            spark
            .read
            .parquet(
                CAMINHO_SILVER + caminho
            )
            .cache()
        )

    except Exception as erro:

        print(
            f"ERRO - {nome}: falha de ingestão"
        )

        print(str(erro))

        logging.error(
            f"{nome}: {erro}"
        )

        continue

    # =================================================
    # VALORES AUSENTES
    # =================================================

    agregacao_nulos = df.select(

        *[

            spark_sum(

                when(
                    col(c).isNull(),
                    1
                ).otherwise(0)

            ).alias(c)

            for c in df.columns

        ]

    ).collect()[0]

    total_nulos = 0

    for coluna in df.columns:

        qtd_nulos = agregacao_nulos[coluna]

        total_nulos += qtd_nulos

        if qtd_nulos > 0:

            print(
                f"ALERTA - {nome}: "
                f"coluna '{coluna}' "
                f"possui {qtd_nulos} nulos"
            )

    if total_nulos == 0:

        print(
            f"{nome}: sem valores nulos"
        )

    # =================================================
    # DUPLICIDADE
    # =================================================

    if nome in CHAVES:

        duplicados = (

            df

            .groupBy(
                CHAVES[nome]
            )

            .count()

            .filter(
                col("count") > 1
            )

            .limit(1)

            .count()

        )

        if duplicados > 0:

            print(
                f"ALERTA - {nome}: "
                f"duplicidade encontrada"
            )

        else:

            print(
                f"{nome}: sem duplicidade"
            )

    # =================================================
    # LATÊNCIA
    # =================================================

    fim_dataset = time.time()

    latencia_dataset = (

        fim_dataset -
        inicio_dataset

    )

    print(
        f"{nome}: latência "
        f"{latencia_dataset:.2f}s"
    )

    df.unpersist()

# ====================================================
# RELACIONAMENTOS
# ====================================================

print("\n" + "=" * 70)
print("VALIDAÇÃO DE RELACIONAMENTOS")
print("=" * 70)

df_uf = spark.read.parquet(
    CAMINHO_SILVER + datasets["uf"]
)

df_municipio = spark.read.parquet(
    CAMINHO_SILVER + datasets["municipio"]
)

df_meta_uf = spark.read.parquet(
    CAMINHO_SILVER + datasets["meta_por_uf"]
)

df_meta_municipio = spark.read.parquet(
    CAMINHO_SILVER + datasets["meta_por_municipio"]
)

# ---------------------------------------------
# UF x META UF
# ---------------------------------------------

possui_problema = (

    df_uf

    .select(
        "ano",
        "sigla_uf"
    )

    .join(

        df_meta_uf.select(
            "ano",
            "sigla_uf"
        ),

        ["ano", "sigla_uf"],

        "left_anti"

    )

    .limit(1)

    .count()

)

if possui_problema:

    print(
        "ALERTA - UF sem meta correspondente"
    )

else:

    print(
        "Relacionamento UF x Meta UF válido"
    )

# ---------------------------------------------
# MUNICÍPIO x META MUNICÍPIO
# ---------------------------------------------

possui_problema = (

    df_municipio

    .select(
        "ano",
        "id_municipio"
    )

    .join(

        df_meta_municipio.select(
            "ano",
            "id_municipio"
        ),

        ["ano", "id_municipio"],

        "left_anti"

    )

    .limit(1)

    .count()

)

if possui_problema:

    print(
        "ALERTA - Município sem meta correspondente"
    )

else:

    print(
        "Relacionamento Município x Meta Município válido"
    )


# ====================================================
# LATÊNCIA TOTAL
# ====================================================

fim_pipeline = time.time()

latencia_total = (

    fim_pipeline -
    inicio_pipeline

)

print("\n" + "=" * 70)

print(
    f"Tempo total do pipeline: "
    f"{latencia_total:.2f}s"
)

print("=" * 70)

StatementMeta(, 33, -1, Cancelled, , Cancelled, True)